# NCKH-UAV: Adversarial Camouflage Attack Pipeline — v6

**Phase 6: Cross-model Transfer Attack**

**Changes from v5:**
- Train **RT-DETR-l** as the target model on the same dataset
- Evaluate YOLOv8n-optimized adversarial patch on RT-DETR (transfer attack)
- Side-by-side comparison: surrogate (YOLOv8n) vs target (RT-DETR)

**Pipeline:** Data prep → Train surrogate (YOLOv8n) → Train target (RT-DETR) → DE attack (on surrogate) → Transfer eval (on target) → Compare

In [ ]:
# ── Cell 1: Environment & GPU ─────────────────────────────
import os, sys, shutil, subprocess, time
from pathlib import Path

# Kaggle dataset mount points
DOTA_INPUT  = '/kaggle/input/datasets/thuyen001/dota-raw'
DIOR_INPUT  = '/kaggle/input/datasets/thuyen001/dior-raw/dior_raw'
CODE_SRC    = '/kaggle/input/datasets/thuyen001/nckh-uav-code/nckh-uav-code'

WORK = Path('/kaggle/working')
DATA = WORK / 'data'

# Copy source tree
dst_src = WORK / 'src'
if dst_src.exists():
    shutil.rmtree(dst_src)
shutil.copytree(os.path.join(CODE_SRC, 'src'), str(dst_src))
shutil.copy2(os.path.join(CODE_SRC, 'configs/base_config.yaml'),
             str(WORK / 'base_config.yaml'))
sys.path.insert(0, str(WORK))

# GPU info
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
print()

In [ ]:
# ── Cell 2: Install packages ──────────────────────────────
!pip install -q ultralytics opencv-python-headless pyyaml tqdm
import ultralytics
print(f'Ultralytics {ultralytics.__version__}')

In [ ]:
# ── Cell 3: Create symlinks & run diagnostics ─────────────
from pathlib import Path
import os

# ── DOTA raw ──
dota_img_src = os.path.join(DOTA_INPUT, 'images')
dota_lbl_src = os.path.join(DOTA_INPUT, 'labelTxt')
dota_dst = DATA / 'DOTA' / 'raw'
dota_dst.mkdir(parents=True, exist_ok=True)

for name, src in [('images', dota_img_src), ('labelTxt', dota_lbl_src)]:
    link = dota_dst / name
    if not link.exists():
        os.symlink(src, str(link))

# ── DIOR raw ──
dior_dst = DATA / 'DIOR' / 'raw'
dior_dst.mkdir(parents=True, exist_ok=True)

for folder_name in ['JPEGImages', 'Annotations']:
    link = dior_dst / folder_name
    src  = os.path.join(DIOR_INPUT, folder_name)
    if os.path.isdir(src) and not link.exists():
        os.symlink(src, str(link))
        print(f'  DIOR symlink: {folder_name} → OK')
    elif not os.path.isdir(src):
        print(f'  DIOR WARNING: {src} not found!')

# Quick check
n_dota_img = len(list(Path(dota_img_src).glob('*.png')))
n_dota_lbl = len(list(Path(dota_lbl_src).glob('*.txt')))
print(f'\nDOTA images: {n_dota_img}  labels: {n_dota_lbl}  matched: {min(n_dota_img,n_dota_lbl)} ✅')

# Check DIOR
dior_jpeg_dir = dior_dst / 'JPEGImages'
dior_ann_dir  = dior_dst / 'Annotations'
if dior_jpeg_dir.exists():
    import glob
    n_dior_img = len(glob.glob(str(dior_jpeg_dir / '*.jpg')))
    n_dior_xml = len(glob.glob(str(dior_ann_dir / '*.xml')))
    print(f'DIOR images: {n_dior_img}  XML labels: {n_dior_xml} ✅')
else:
    print('DIOR: JPEGImages not found — check DIOR_INPUT path')

In [ ]:
# ── Cell 4: Quick GPU test ────────────────────────────────
from ultralytics import YOLO
import numpy as np, cv2

dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
tmp_path = '/tmp/gpu_test.jpg'
cv2.imwrite(tmp_path, dummy)

m = YOLO('yolov8n.pt')
r = m.predict(source=tmp_path, device='0', verbose=False)
print(f'Dummy GPU test passed — {len(r[0].boxes)} detections on noise image')

In [ ]:
# ── Cell 5: Process DOTA (PIL slicing) + DIOR (XML→YOLO) ─
import glob, cv2, numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm

Image.MAX_IMAGE_PIXELS = None  # large DOTA images

# ═══════════════════════════════════════════════════════════
# Part A: DOTA slicing (unchanged from v4)
# ═══════════════════════════════════════════════════════════
DOTA_CLASSES = {'plane': 0, 'small-vehicle': 1, 'large-vehicle': 1}
SLICE_SIZE = 1024
OVERLAP = 0.2

dota_out_img = DATA / 'DOTA' / 'processed' / 'images'
dota_out_lbl = DATA / 'DOTA' / 'processed' / 'labels'
dota_out_img.mkdir(parents=True, exist_ok=True)
dota_out_lbl.mkdir(parents=True, exist_ok=True)

dota_raw_imgs = sorted(glob.glob(str(DATA / 'DOTA' / 'raw' / 'images' / '*.png')))
dota_raw_lbls = str(DATA / 'DOTA' / 'raw' / 'labelTxt')

def parse_dota_label(label_path):
    anns = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 9: continue
            cls_name = parts[8]
            if cls_name not in DOTA_CLASSES: continue
            coords = list(map(float, parts[:8]))
            xs = coords[0::2]
            ys = coords[1::2]
            anns.append((min(xs), min(ys), max(xs), max(ys), DOTA_CLASSES[cls_name]))
    return anns

total_slices = 0
read_ok = 0
read_fail = 0

for img_path in tqdm(dota_raw_imgs, desc='DOTA slicing'):
    stem = Path(img_path).stem
    lbl_path = os.path.join(dota_raw_lbls, stem + '.txt')
    if not os.path.isfile(lbl_path): continue
    anns = parse_dota_label(lbl_path)
    if len(anns) == 0: continue

    try:
        pil_img = Image.open(img_path)
        img_np = np.array(pil_img)
        if img_np.ndim == 2:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_GRAY2BGR)
        elif img_np.shape[2] == 4:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGBA2BGR)
        else:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        read_ok += 1
    except Exception as e:
        read_fail += 1
        continue

    h, w = img_np.shape[:2]
    stride = int(SLICE_SIZE * (1 - OVERLAP))

    for y0 in range(0, h, stride):
        for x0 in range(0, w, stride):
            x1 = min(x0 + SLICE_SIZE, w)
            y1 = min(y0 + SLICE_SIZE, h)
            x0a = x1 - SLICE_SIZE
            y0a = y1 - SLICE_SIZE
            if x0a < 0: x0a = 0
            if y0a < 0: y0a = 0

            slice_anns = []
            for (ax1, ay1, ax2, ay2, cls_id) in anns:
                ix1 = max(ax1, x0a)
                iy1 = max(ay1, y0a)
                ix2 = min(ax2, x1)
                iy2 = min(ay2, y1)
                iw = ix2 - ix1
                ih = iy2 - iy1
                if iw > 10 and ih > 10:
                    ow = ax2 - ax1
                    oh = ay2 - ay1
                    if (iw * ih) / max(ow * oh, 1) > 0.3:
                        sw = x1 - x0a
                        sh = y1 - y0a
                        cx = ((ix1 - x0a) + (ix2 - x0a)) / 2 / sw
                        cy = ((iy1 - y0a) + (iy2 - y0a)) / 2 / sh
                        bw = iw / sw
                        bh = ih / sh
                        slice_anns.append(f'{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

            if len(slice_anns) == 0: continue

            slice_img = img_np[y0a:y1, x0a:x1]
            slice_name = f'{stem}_{x0a}_{y0a}'
            cv2.imwrite(str(dota_out_img / f'{slice_name}.jpg'), slice_img)
            with open(str(dota_out_lbl / f'{slice_name}.txt'), 'w') as f:
                f.write('\n'.join(slice_anns) + '\n')
            total_slices += 1

print(f'\nDOTA: {read_ok} read OK, {read_fail} failed, {total_slices} slices')

# ═══════════════════════════════════════════════════════════
# Part B: DIOR — Convert VOC XML → YOLO format
# ═══════════════════════════════════════════════════════════
from src.data_prep.process_dior import main as process_dior_main

dior_proc_img = DATA / 'DIOR' / 'processed' / 'images'
dior_proc_lbl = DATA / 'DIOR' / 'processed' / 'labels'
dior_proc_img.mkdir(parents=True, exist_ok=True)
dior_proc_lbl.mkdir(parents=True, exist_ok=True)

dior_jpeg_dir = DATA / 'DIOR' / 'raw' / 'JPEGImages'
dior_ann_dir  = DATA / 'DIOR' / 'raw' / 'Annotations'

if dior_ann_dir.exists():
    n_dior = process_dior_main(
        input_img_dir=str(dior_jpeg_dir),
        input_xml_dir=str(dior_ann_dir),
        out_img_dir=str(dior_proc_img),
        out_lbl_dir=str(dior_proc_lbl),
        use_symlink=True,
    )
    print(f'\nDIOR: {n_dior} images converted (XML → YOLO, symlinked)')
else:
    n_dior = 0
    print(f'DIOR: Annotations dir not found at {dior_ann_dir} — skipping')

print(f'\n══ Summary: DOTA {total_slices} slices + DIOR {n_dior} images ══')

In [ ]:
# ── Cell 6: Train/Val split + data.yaml ───────────────────
import random, shutil

SPLIT_RATIO = 0.8
SEED = 42

yolo_root = DATA / 'yolo_dataset'
for sub in ['train/images','train/labels','val/images','val/labels']:
    (yolo_root / sub).mkdir(parents=True, exist_ok=True)

def collect_pairs(img_dir, lbl_dir):
    pairs = []
    for ip in Path(img_dir).iterdir():
        if ip.suffix.lower() not in ('.jpg','.jpeg','.png','.bmp'): continue
        lp = Path(lbl_dir) / (ip.stem + '.txt')
        if lp.exists() and lp.stat().st_size > 5:
            pairs.append((str(ip), str(lp)))
    return pairs

dota_pairs = collect_pairs(dota_out_img, dota_out_lbl)
dior_pairs = collect_pairs(dior_proc_img, dior_proc_lbl)
all_pairs = dota_pairs + dior_pairs
print(f'DOTA {len(dota_pairs)} + DIOR {len(dior_pairs)} = {len(all_pairs)} pairs')

random.seed(SEED)
random.shuffle(all_pairs)
split = int(len(all_pairs) * SPLIT_RATIO)
train_pairs = all_pairs[:split]
val_pairs   = all_pairs[split:]
print(f'Train {len(train_pairs)} / Val {len(val_pairs)}')

def symlink_pairs(pairs, img_dst, lbl_dst):
    for ip, lp in pairs:
        di = Path(img_dst) / Path(ip).name
        dl = Path(lbl_dst) / Path(lp).name
        if not di.exists(): os.symlink(ip, str(di))
        if not dl.exists(): os.symlink(lp, str(dl))

symlink_pairs(train_pairs, yolo_root/'train/images', yolo_root/'train/labels')
symlink_pairs(val_pairs,   yolo_root/'val/images',   yolo_root/'val/labels')

# data.yaml
data_yaml = yolo_root / 'data.yaml'
data_yaml.write_text(
    f'path: {yolo_root}\n'
    f'train: train/images\n'
    f'val: val/images\n'
    f'nc: 2\n'
    f'names: ["aircraft", "vehicle"]\n'
)
print(f'data.yaml → {data_yaml}')

In [ ]:
# ── Cell 7: Train YOLOv8n surrogate ───────────────────────
from ultralytics import YOLO

EPOCHS = 50
IMGSZ  = 1024
BATCH  = 4

model_surr = YOLO('yolov8n.pt')
results_surr = model_surr.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device='0,1',
    project=str(WORK / 'runs/detect'),
    name='surrogate_yolov8n',
    exist_ok=True,
    patience=10,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)

SURR_DIR = WORK / 'runs/detect/surrogate_yolov8n'
SURR_PT  = SURR_DIR / 'weights' / 'best.pt'
print(f'\n✅ Surrogate (YOLOv8n) training complete.')
print(f'   Best weights: {SURR_PT}  exists={SURR_PT.exists()}')

In [ ]:
# ── Cell 8: ★ Train RT-DETR-l Target Model ★ ─────────────
from ultralytics import YOLO

RTDETR_EPOCHS = 50
RTDETR_IMGSZ  = 640    # RT-DETR is heavier → reduce to 640 to fit T4 memory
RTDETR_BATCH  = 4      # may need to reduce if OOM

print('\n' + '═' * 60)
print('  TRAINING TARGET MODEL: RT-DETR-l')
print('═' * 60)

model_target = YOLO('rtdetr-l.pt')
results_target = model_target.train(
    data=str(data_yaml),
    epochs=RTDETR_EPOCHS,
    imgsz=RTDETR_IMGSZ,
    batch=RTDETR_BATCH,
    device='0,1',
    project=str(WORK / 'runs/detect'),
    name='target_rtdetr',
    exist_ok=True,
    patience=10,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)

TARGET_DIR = WORK / 'runs/detect/target_rtdetr'
TARGET_PT  = TARGET_DIR / 'weights' / 'best.pt'
print(f'\n✅ Target (RT-DETR-l) training complete.')
print(f'   Best weights: {TARGET_PT}  exists={TARGET_PT.exists()}')

In [ ]:
# ── Cell 9: ★ DE Attack on YOLOv8n Surrogate ★ ───────────
import sys, os
sys.path.insert(0, str(WORK))

from src.attack.de_attack_real import DEAttackReal, plot_convergence, plot_patch, save_attack_examples

# ── Config ─────────────────────────────────────────────────
SURR_PT    = WORK / 'runs/detect/surrogate_yolov8n/weights/best.pt'
VAL_IMAGES = DATA / 'yolo_dataset' / 'val' / 'images'
VAL_LABELS = DATA / 'yolo_dataset' / 'val' / 'labels'

PATCH_SIZE   = 32
NUM_PATCHES  = 1
POP_SIZE     = 20
MAX_GEN      = 50
F  = 0.8
CR = 0.7

W_OBJ = 1.0
W_FP  = 0.3
W_TV  = 0.01
W_NPS = 0.001

print(f'Surrogate weights: {SURR_PT}  (exists={SURR_PT.exists()})')
print()

# ── Initialise attacker (YOLOv8n) ─────────────────────────
attacker_surr = DEAttackReal(
    model_path=str(SURR_PT),
    device='0',
    conf_threshold=0.25,
    iou_threshold=0.5,
    verbose=True,
)

n_ready = attacker_surr.prepare_attack_set(
    img_dir=str(VAL_IMAGES),
    lbl_dir=str(VAL_LABELS),
    n_images=30,
    seed=42,
)
assert n_ready >= 10, f'Only {n_ready} images with detections — need ≥10'

# ── Run DE optimisation ──────────────────────────────────
best_patch, history = attacker_surr.run_de(
    patch_size=PATCH_SIZE,
    num_patches=NUM_PATCHES,
    pop_size=POP_SIZE,
    max_gen=MAX_GEN,
    F=F, CR=CR,
    w_obj=W_OBJ, w_fp=W_FP, w_tv=W_TV, w_nps=W_NPS,
)

# ── Save artefacts ───────────────────────────────────────
import numpy as np
ATTACK_DIR = WORK / 'attack_results'
ATTACK_DIR.mkdir(exist_ok=True)

np.save(str(ATTACK_DIR / 'best_patch.npy'), best_patch)
np.save(str(ATTACK_DIR / 'de_history.npy'), np.array(history))

plot_convergence(history, str(ATTACK_DIR / 'de_convergence.png'))
plot_patch(best_patch,   str(ATTACK_DIR / 'best_patch_vis.png'))

print(f'\nPatch shape: {best_patch.shape}, mean={best_patch.mean():.1f}, std={best_patch.std():.1f}')
print(f'Artefacts saved to {ATTACK_DIR}')

In [ ]:
# ── Cell 10: ★ Cross-Model Transfer Evaluation ★ ─────────
import numpy as np
from src.attack.de_attack_real import DEAttackReal, save_attack_examples

TARGET_PT = WORK / 'runs/detect/target_rtdetr/weights/best.pt'
SURR_PT   = WORK / 'runs/detect/surrogate_yolov8n/weights/best.pt'

best_patch = np.load(str(ATTACK_DIR / 'best_patch.npy'))

# ═══════════════════════════════════════════════════════════
# Part A: Evaluate on SURROGATE (YOLOv8n) — baseline
# ═══════════════════════════════════════════════════════════
print('═' * 60)
print('  A) SURROGATE — YOLOv8n (patch was optimised against this)')
print('═' * 60)

attacker_surr = DEAttackReal(
    model_path=str(SURR_PT), device='0',
    conf_threshold=0.25, iou_threshold=0.5, verbose=True,
)

# In-sample
print('\n── In-sample (30 imgs, seed=42) ──')
attacker_surr.prepare_attack_set(
    img_dir=str(VAL_IMAGES), lbl_dir=str(VAL_LABELS),
    n_images=30, seed=42,
)
m_surr_in = attacker_surr.evaluate_patch(best_patch, num_patches=NUM_PATCHES)
for k, v in m_surr_in.items():
    print(f'  {k:>22s} : {v}')

# Held-out
print('\n── Held-out (50 imgs, seed=99) ──')
m_surr_out = attacker_surr.evaluate_patch(
    best_patch, img_dir=str(VAL_IMAGES), lbl_dir=str(VAL_LABELS),
    n_images=50, num_patches=NUM_PATCHES, seed=99,
)
for k, v in m_surr_out.items():
    print(f'  {k:>22s} : {v}')

# Save surrogate examples
save_attack_examples(
    attacker_surr, best_patch, num_patches=NUM_PATCHES,
    save_dir=str(ATTACK_DIR / 'examples_yolov8n'), n=6,
)

# ═══════════════════════════════════════════════════════════
# Part B: Evaluate on TARGET (RT-DETR) — TRANSFER ATTACK
# ═══════════════════════════════════════════════════════════
print('\n')
print('═' * 60)
print('  B) TARGET — RT-DETR-l (transfer attack, never saw patch)')
print('═' * 60)

attacker_target = DEAttackReal(
    model_path=str(TARGET_PT), device='0',
    conf_threshold=0.25, iou_threshold=0.5, verbose=True,
)

# In-sample (same 30 images, but detections come from RT-DETR)
print('\n── In-sample (30 imgs, seed=42) ──')
n_target_in = attacker_target.prepare_attack_set(
    img_dir=str(VAL_IMAGES), lbl_dir=str(VAL_LABELS),
    n_images=30, seed=42,
)
if n_target_in >= 5:
    m_target_in = attacker_target.evaluate_patch(best_patch, num_patches=NUM_PATCHES)
    for k, v in m_target_in.items():
        print(f'  {k:>22s} : {v}')
else:
    m_target_in = {'ASR': 0, 'AORR': 0, 'targets_total': 0}
    print(f'  ⚠ Only {n_target_in} images with RT-DETR detections')

# Held-out
print('\n── Held-out (50 imgs, seed=99) ──')
m_target_out = attacker_target.evaluate_patch(
    best_patch, img_dir=str(VAL_IMAGES), lbl_dir=str(VAL_LABELS),
    n_images=50, num_patches=NUM_PATCHES, seed=99,
)
for k, v in m_target_out.items():
    print(f'  {k:>22s} : {v}')

# Save RT-DETR examples
save_attack_examples(
    attacker_target, best_patch, num_patches=NUM_PATCHES,
    save_dir=str(ATTACK_DIR / 'examples_rtdetr'), n=6,
)

# ═══════════════════════════════════════════════════════════
# Part C: Summary comparison table
# ═══════════════════════════════════════════════════════════
print('\n')
print('═' * 70)
print('  CROSS-MODEL TRANSFER SUMMARY')
print('═' * 70)
print(f'{"Metric":>25s} | {"YOLOv8n (surr)":>16s} | {"RT-DETR (target)":>16s} | {"Transfer gap":>14s}')
print('-' * 76)

rows = [
    ('ASR in-sample', m_surr_in.get('ASR',0), m_target_in.get('ASR',0)),
    ('ASR held-out',  m_surr_out.get('ASR',0), m_target_out.get('ASR',0)),
    ('AORR held-out', m_surr_out.get('AORR',0), m_target_out.get('AORR',0)),
    ('New FP held-out', m_surr_out.get('avg_new_fp',0), m_target_out.get('avg_new_fp',0)),
    ('Targets held-out', m_surr_out.get('targets_total',0), m_target_out.get('targets_total',0)),
]

for label, vs, vt in rows:
    if isinstance(vs, float) and isinstance(vt, float):
        gap = vt - vs
        print(f'{label:>25s} | {vs:>16.4f} | {vt:>16.4f} | {gap:>+14.4f}')
    else:
        print(f'{label:>25s} | {str(vs):>16s} | {str(vt):>16s} | {"":>14s}')

print('-' * 76)
surr_asr  = m_surr_out.get('ASR', 0)
tgt_asr   = m_target_out.get('ASR', 0)
if surr_asr > 0:
    transfer_rate = tgt_asr / surr_asr * 100
else:
    transfer_rate = 0.0
print(f'\n  Transfer rate = ASR_target / ASR_surrogate = {transfer_rate:.1f}%')
print(f'  (100% = perfect transfer, 0% = no transfer)')

print(f'\n✅ Phase 6 complete.')

In [ ]:
# ── Cell 11: Display all results ─────────────────────────
from IPython.display import display, Image as IPImage
from pathlib import Path
import os

SURR_DIR   = WORK / 'runs/detect/surrogate_yolov8n'
TARGET_DIR = WORK / 'runs/detect/target_rtdetr'
ATTACK_DIR = WORK / 'attack_results'

# ── Surrogate Training curves ─────────────────────────────
print('═' * 60)
print('  Surrogate (YOLOv8n) — Training Results')
print('═' * 60)
for name in ['results.png', 'confusion_matrix.png']:
    p = SURR_DIR / name
    if p.exists():
        print(f'\n{name}:')
        display(IPImage(filename=str(p), width=700))

# ── Target Training curves ────────────────────────────────
print('\n')
print('═' * 60)
print('  Target (RT-DETR-l) — Training Results')
print('═' * 60)
for name in ['results.png', 'confusion_matrix.png']:
    p = TARGET_DIR / name
    if p.exists():
        print(f'\n{name}:')
        display(IPImage(filename=str(p), width=700))

# ── Attack results ───────────────────────────────────────
print('\n')
print('═' * 60)
print('  DE Attack Convergence & Patch')
print('═' * 60)
for name in ['de_convergence.png', 'best_patch_vis.png']:
    p = ATTACK_DIR / name
    if p.exists():
        print(f'\n{name}:')
        display(IPImage(filename=str(p), width=600))

# ── Side-by-side: YOLOv8n ─────────────────────────────────
print('\n')
print('═' * 60)
print('  Attack Examples — YOLOv8n (surrogate)')
print('  green = clean detections | red = after patch')
print('═' * 60)
yolo_ex = ATTACK_DIR / 'examples_yolov8n'
if yolo_ex.exists():
    for f in sorted(yolo_ex.glob('*.jpg')):
        display(IPImage(filename=str(f), width=900))

# ── Side-by-side: RT-DETR ─────────────────────────────────
print('\n')
print('═' * 60)
print('  Attack Examples — RT-DETR-l (target, transfer attack)')
print('  green = clean detections | red = after patch')
print('═' * 60)
rtdetr_ex = ATTACK_DIR / 'examples_rtdetr'
if rtdetr_ex.exists():
    for f in sorted(rtdetr_ex.glob('*.jpg')):
        display(IPImage(filename=str(f), width=900))

print('\n✅ All v6 results displayed.')